<a href="https://colab.research.google.com/github/njwbilll/Tugas-2_scikit-learn-Cookbook-O-Reilly-_Najwa-Bilqis-Al-Khalidah/blob/main/01_Common_Conventions_and_API_Elements.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1: Common Conventions and API Elements of scikit-learn

**Referensi:** scikit-learn Cookbook, Third Edition - John Sukup (Packt Publishing, 2025)

---

## Ringkasan Chapter

Chapter 1 memperkenalkan konvensi standar dan elemen inti API scikit-learn, termasuk filosofi desain di balik estimators, transformers, dan pipelines, serta method umum seperti `fit()`, `predict()`, dan `transform()`. Chapter ini menjadi fondasi bagi seluruh pembahasan dalam buku ini.

### Topik yang Dibahas:
1. Introduction to scikit-learn's design philosophy
2. Understanding estimators
3. Transformers and the `transform()` method
4. Handling custom estimators and transformers
5. Pipelines and workflow automation
6. Common attributes and methods
7. Hyperparameter tuning with search methods
8. Working with metadata: Tags and more
9. Best practices for API usage


---
## 1. Filosofi Desain scikit-learn

### Penjelasan Teori

scikit-learn dirancang di atas empat prinsip inti:

1. **Consistency (Konsistensi):** Semua model mengikuti pola yang sama, yaitu `fit()` untuk melatih model, `predict()` untuk membuat prediksi, dan `transform()` untuk memanipulasi data. Konsistensi ini memungkinkan pengguna untuk dengan mudah beralih antar model.

2. **Simplicity (Kesederhanaan):** API dirancang agar mudah dipelajari dan digunakan, bahkan oleh pemula sekalipun.

3. **Modularity (Modularitas):** Komponen seperti estimators, transformers, dan pipelines dapat dikombinasikan dan digunakan kembali di berbagai tugas. Ini mendorong penggunaan ulang kode (software reuse).

4. **Reusability (Penggunaan ulang):** Langkah-langkah preprocessing seperti scaling dan encoding dapat diintegrasikan langsung ke dalam proses pemodelan menggunakan kelas `Pipeline()`.

> **Catatan:** Penulisan yang benar adalah `scikit-learn` (huruf kecil semua). Pengucapannya adalah "sy-kit", di mana "sci" adalah singkatan dari kata "science".


---
## 2. Memahami Estimators

### Penjelasan Teori

**Estimator** adalah objek inti dalam scikit-learn yang mengimplementasikan algoritma pembelajaran dari data. Setiap estimator, baik model maupun transformer, mengikuti antarmuka yang sederhana dan intuitif.

**Dua method paling esensial dari setiap estimator:**

- `fit(X, y)`: Melatih model dengan cara mempelajari pola dari data. Pada regresi linear, ini berarti model mencari koefisien optimal.
- `predict(X)`: Menghasilkan prediksi pada data baru berdasarkan model yang sudah dilatih.

**Kenapa ada `fit()` dan `predict()` secara terpisah?**

Pemisahan ini penting karena:
- `fit()` diterapkan pada data **training** untuk mempelajari parameter.
- `predict()` diterapkan pada data **testing/baru** yang tidak dilihat model saat training.

Metode `fit_predict()` menggabungkan keduanya dalam satu panggilan API dan biasanya digunakan dalam **unsupervised learning** di mana tidak ada target variable di data training.


In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression

# Contoh data
X = np.array([[1], [2], [3], [4], [5]])  # Feature matrix
y = np.array([1, 2, 3, 3.5, 5])          # Target values

# Membuat dan melatih model
model = LinearRegression()
model.fit(X, y)

# Prediksi pada data baru
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print("Prediksi untuk X=[6,7]:", predictions)


Prediksi untuk X=[6,7]: [5.75 6.7 ]


In [2]:
# Contoh fit_predict() pada KMeans (unsupervised learning)
from sklearn.cluster import KMeans

X_cluster = np.array([[1], [2], [3], [4], [5]])

kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
labels = kmeans.fit_predict(X_cluster)
print("Label cluster untuk setiap data point:", labels)


Label cluster untuk setiap data point: [0 0 0 1 1]


---
## 3. Transformers dan Method `transform()`

### Penjelasan Teori

**Transformer** adalah alat dalam scikit-learn yang memodifikasi data melalui transformasi seperti scaling, normalisasi, atau encoding. Transformer mengikuti antarmuka yang konsisten:

- `fit(X)`: Mempelajari parameter yang diperlukan dari data (misalnya, menghitung mean dan standar deviasi).
- `transform(X)`: Menerapkan transformasi tersebut pada data.
- `fit_transform(X)`: Menggabungkan kedua langkah di atas menjadi satu pemanggilan.

**Kapan menggunakan `fit_transform()` vs `fit()` + `transform()` secara terpisah?**

- `fit_transform()` digunakan pada **data training** ketika kita ingin langsung mentransformasi data berdasarkan parameter yang dihitung.
- Pada **data testing**, kita hanya menggunakan `transform()` (TIDAK `fit()` ulang), karena kita harus menggunakan parameter yang sama dengan yang dipelajari dari data training. Melakukan `fit()` ulang pada data testing dapat menyebabkan **data leakage** dan membuat prediksi model tidak dapat diandalkan.

**Contoh:** `StandardScaler()` menghitung mean dan standar deviasi dari data training saat `fit()`, lalu menggunakan nilai tersebut untuk mentransformasi data (menghasilkan Z-score).


In [3]:
from sklearn.preprocessing import StandardScaler

# Data contoh
X = np.array([[1, 2], [3, 4], [5, 6]])

# Menggunakan fit() dan transform() secara terpisah
scaler = StandardScaler()
scaler.fit(X)                       # Pelajari mean dan std dari data
X_scaled_manual = scaler.transform(X)  # Terapkan transformasi
print("Hasil fit() + transform():")
print(X_scaled_manual)


Hasil fit() + transform():
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


In [4]:
# Menggunakan fit_transform() langsung (ekuivalen untuk data training)
scaler2 = StandardScaler()
X_scaled_combined = scaler2.fit_transform(X)
print("Hasil fit_transform():")
print(X_scaled_combined)


Hasil fit_transform():
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]


---
## 4. Custom Estimators dan Transformers

### Penjelasan Teori

API scikit-learn dirancang agar dapat diperluas (extensible). Pengembang dapat membuat estimators dan transformers kustom yang terintegrasi dengan mulus ke dalam alur kerja yang sudah ada.

**Cara membuat custom estimator/transformer:**

1. Subclass `BaseEstimator` dan mixin classes yang sesuai (misalnya `TransformerMixin` untuk transformer, `ClassifierMixin` untuk classifier).
2. Implementasikan method `fit()` dan `transform()` (untuk transformer) atau `fit()` dan `predict()` (untuk model).
3. Pastikan kompatibilitas dengan tools seperti `GridSearchCV()` dan `Pipeline()`.

**Mixin Classes:** Mixin adalah cara untuk memperluas fungsionalitas kelas tanpa menggunakan inheritance tradisional. Mixin berguna untuk code reusability, memungkinkan programmer berbagi fungsionalitas antar kelas yang berbeda.

`check_is_fitted()` digunakan untuk memvalidasi apakah estimator sudah di-fit sebelum digunakan untuk prediksi atau transformasi.


In [5]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

# Contoh Custom Transformer: menambahkan nilai konstan ke setiap fitur
class ConstantAdder(BaseEstimator, TransformerMixin):
    def __init__(self, constant=1):
        self.constant = constant

    def fit(self, X, y=None):
        # Tidak perlu mempelajari parameter dari data untuk transformer sederhana ini
        self.n_features_in_ = X.shape[1]  # Diperlukan untuk check_is_fitted
        self.fitted_ = True
        return self  # Selalu kembalikan self dari fit()

    def transform(self, X, y=None):
        check_is_fitted(self)
        return X + self.constant

# Penggunaan
X_demo = np.array([[1, 2], [3, 4], [5, 6]])
adder = ConstantAdder(constant=10)
adder.fit(X_demo)
result = adder.transform(X_demo)
print("Data asli:")
print(X_demo)
print("Setelah ConstantAdder(constant=10):")
print(result)


Data asli:
[[1 2]
 [3 4]
 [5 6]]
Setelah ConstantAdder(constant=10):
[[11 12]
 [13 14]
 [15 16]]


---
## 5. Pipelines dan Workflow Automation

### Penjelasan Teori

**Pipeline** dalam scikit-learn menyediakan cara terstruktur untuk mengotomatisasi alur kerja ML dengan menghubungkan beberapa langkah pemrosesan, seperti preprocessing data, pelatihan model, dan prediksi, menjadi satu objek yang kohesif.

**Keuntungan utama menggunakan Pipeline:**

1. **Sequential execution:** Setiap langkah dieksekusi dalam urutan yang ditentukan, memastikan transformasi diterapkan secara konsisten pada data training dan testing.
2. **Code simplification:** Pipeline mengkondensasikan banyak baris kode menjadi satu objek, membuat basis kode lebih bersih.
3. **Consistency:** Memastikan transformasi yang sama diterapkan saat training dan prediksi, meminimalkan risiko data leakage.
4. **Easier hyperparameter tuning:** Pipeline terintegrasi dengan `GridSearchCV` untuk mengoptimalkan parameter.
5. **Modularity:** Mendorong modularitas dengan memungkinkan tahapan pemrosesan yang berbeda dienkapsulasi sebagai komponen yang dapat digunakan kembali.

**Keterkaitan Pipeline dengan MLOps:** scikit-learn mendukung workflow MLOps melalui kelas `Pipeline()`, `GridSearchCV()`, serta library seperti `joblib` dan `pickle` untuk menyimpan dan mendeploy model.


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

# Contoh Pipeline: scaling -> PCA -> model
pipeline_demo = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=1)),
    ('model', LinearRegression())
])

# Data contoh
X_pipe = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]])
y_pipe = np.array([1, 2, 3, 4, 5])

pipeline_demo.fit(X_pipe, y_pipe)
preds = pipeline_demo.predict(X_pipe)
print("Prediksi dari pipeline:", preds)


Prediksi dari pipeline: [1. 2. 3. 4. 5.]


In [7]:
# Visualisasi struktur pipeline (jika dijalankan di Jupyter)
from sklearn import set_config
set_config(display='diagram')
pipeline_demo


Pipeline(steps=[('scaler', StandardScaler()), ('pca', PCA(n_components=1)),
                ('model', LinearRegression())])

---
## 6. Atribut dan Method Umum

### Penjelasan Teori

Model-model scikit-learn berbagi beberapa atribut dan method kunci yang memberikan wawasan tentang bagaimana model belajar dari data:

**Atribut umum (diakses setelah `fit()`):**
- `coef_`: Koefisien (bobot) yang dipelajari untuk model linear.
- `intercept_`: Nilai intercept (bias) untuk model linear.
- `feature_importances_`: Tingkat kepentingan setiap fitur (untuk model berbasis pohon).
- `classes_`: Label kelas yang diketahui (untuk classifier).

**Method umum:**
- `score(X, y)`: Mengevaluasi performa model. Default metric: akurasi untuk classifier, R2 untuk regressor.
- `get_params()`: Mengambil hyperparameter model saat ini.
- `set_params(**params)`: Mengubah hyperparameter model secara programatik.

> **Catatan tentang R-squared (R2):** R2 mengukur seberapa baik model menjelaskan variasi dalam data target. Nilai R2 mendekati 1 berarti model sangat baik. Namun, R2 dapat menyesatkan karena akan selalu meningkat saat lebih banyak variabel ditambahkan. Seringkali, **adjusted R2** lebih disarankan.


In [8]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Data contoh
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 2, 3, 3.5, 5])

model = LinearRegression()
model.fit(X, y)

# Mengakses atribut model setelah training
print("Koefisien (slope):", model.coef_)
print("Intercept (y-intercept):", model.intercept_)

# Method score() untuk evaluasi model (mengembalikan R-squared)
r_squared = model.score(X, y)
print("R-squared score:", round(r_squared, 4))


Koefisien (slope): [0.95]
Intercept (y-intercept): 0.04999999999999938
R-squared score: 0.981


---
## 7. Hyperparameter Tuning dengan Search Methods

### Penjelasan Teori

**Hyperparameter** adalah parameter model yang ditetapkan sebelum training (bukan dipelajari dari data). Tuning hyperparameter sangat penting untuk mengoptimalkan performa model.

scikit-learn menyediakan beberapa metode untuk tuning hyperparameter:

1. **`GridSearchCV()`:** Mencoba semua kombinasi hyperparameter yang ditentukan secara exhaustif. Cocok ketika ruang pencarian kecil.

2. **`RandomizedSearchCV()`:** Mengambil sampel acak dari distribusi hyperparameter yang ditentukan. Lebih efisien untuk ruang pencarian yang besar.

3. **`set_params()` dan `get_params()`:** Pendekatan manual untuk menyetel dan memeriksa hyperparameter model. `set_params()` mengubah hyperparameter secara programatik, sedangkan `get_params()` mengambil pengaturan hyperparameter saat ini.

**Successive Halving:** scikit-learn juga menyediakan `HalvingGridSearchCV` dan `HalvingRandomSearchCV` yang mengimplementasikan pendekatan successive halving, di mana kandidat yang buruk dieliminasi lebih awal untuk efisiensi komputasi.


In [9]:
from sklearn.ensemble import RandomForestClassifier

# Membuat model
model_rf = RandomForestClassifier()

# Menyetel hyperparameter menggunakan set_params()
model_rf.set_params(n_estimators=100, max_depth=10, random_state=42)

# Memeriksa semua hyperparameter menggunakan get_params()
params = model_rf.get_params()
print("Semua hyperparameter model:")
for key, val in params.items():
    print(f"  {key}: {val}")


Semua hyperparameter model:
  bootstrap: True
  ccp_alpha: 0.0
  class_weight: None
  criterion: gini
  max_depth: 10
  max_features: sqrt
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 1
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 100
  n_jobs: None
  oob_score: False
  random_state: 42
  verbose: 0
  warm_start: False


In [10]:
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import load_iris
from sklearn.svm import SVC

# Load dataset
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

# Definisikan grid hyperparameter
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

# GridSearchCV: mencoba semua kombinasi
svc = SVC(random_state=42)
grid_search = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_iris, y_iris)

print("Hyperparameter terbaik:", grid_search.best_params_)
print("Akurasi terbaik (CV):", round(grid_search.best_score_, 4))


Hyperparameter terbaik: {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
Akurasi terbaik (CV): 0.98


---
## 8. Working with Metadata: Tags dan Lebih

### Penjelasan Teori

scikit-learn menggunakan **metadata**, seperti estimator tags, untuk mengontrol bagaimana model berperilaku dalam berbagai konteks, termasuk cross-validation dan pemrosesan pipeline.

**Dua varietas objek metadata:**
- **Routers:** Memindahkan metadata ke consumers.
- **Consumers:** Menggunakan metadata tersebut dalam perhitungan mereka.

Ini dikenal sebagai **metadata routing** dalam scikit-learn.

**Kegunaan Metadata Routing:** Memungkinkan pengguna mengontrol bagaimana metadata dilewatkan antar objek dalam pipeline. Contohnya, dalam dataset yang tidak seimbang (imbalanced), metadata routing dapat digunakan untuk melewatkan sample weights ke transformer dan classifier tertentu, sementara mengabaikannya di langkah lain seperti scaling.

**Contoh tag estimator:**
- Apakah estimator dapat menangani data multi-output?
- Apakah estimator dapat menangani missing values?
- Tipe input yang didukung.


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.utils.estimator_checks import parametrize_with_checks

# Melihat tags dari sebuah estimator
lr = LogisticRegression()

# Di scikit-learn modern, tags diakses via __sklearn_tags__()
tags = lr.__sklearn_tags__()
print("Tags dari LogisticRegression:")
print(f"  Classifier: {tags.classifier_tags is not None}")
print(f"  Supports sample weights: {tags.classifier_tags.multi_label if tags.classifier_tags else 'N/A'}")
print(f"  Estimator type: {type(tags).__name__}")


Tags dari LogisticRegression:
  Classifier: True
  Supports sample weights: False
  Estimator type: Tags


---
## 9. Best Practices untuk Penggunaan API

### Penjelasan Teori

Berikut adalah praktik terbaik saat bekerja dengan API scikit-learn:

1. **Uniform API:** Semua estimator mengikuti pola dasar yang sama: `fit()`, `transform()` (untuk transformers), dan `predict()`, membuat kode lebih mudah dibaca dan dikembangkan.

2. **Data Preprocessing:** Selalu lakukan preprocessing data menggunakan tools yang tepat dari `sklearn.preprocessing` sebelum memasukkannya ke model.

3. **Gunakan Pipelines:** Untuk workflow yang melibatkan banyak transformasi dan model, gunakan `Pipeline()` untuk menghubungkan operasi, menyederhanakan kode, dan mengelola hyperparameter tuning.

4. **Cross-validation:** Evaluasi performa model menggunakan teknik cross-validation dari `sklearn.model_selection` untuk mendapatkan estimasi generalisasi yang andal.

5. **Hyperparameter Tuning:** Gunakan `GridSearchCV()` atau `RandomizedSearchCV()` untuk menemukan hyperparameter optimal.

6. **Pisahkan data sebelum transformasi:** Selalu lakukan `train_test_split` sebelum menerapkan transformasi apa pun untuk menghindari data leakage.


In [12]:
# Contoh komprehensif: menerapkan best practices
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris

# 1. Load data
iris = load_iris()
X, y = iris.data, iris.target

# 2. Pisahkan data SEBELUM transformasi (mencegah data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Buat Pipeline (preprocessing + model)
pipeline_best = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# 4. Latih pipeline pada data training
pipeline_best.fit(X_train, y_train)

# 5. Evaluasi pada data testing
test_accuracy = pipeline_best.score(X_test, y_test)
print(f"Akurasi pada data testing: {test_accuracy:.4f}")

# 6. Evaluasi lebih andal dengan cross-validation
cv_scores = cross_val_score(pipeline_best, X_train, y_train, cv=5)
print(f"Cross-validation scores: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")


Akurasi pada data testing: 1.0000
Cross-validation scores: [0.95833333 1.         0.875      1.         0.95833333]
Mean CV accuracy: 0.9583 (+/- 0.0456)


---
## Ringkasan Chapter 1

| Konsep | Deskripsi Singkat |
|--------|-------------------|
| Estimator | Objek yang mengimplementasikan algoritma ML; method utama: `fit()` dan `predict()` |
| Transformer | Memodifikasi data; method utama: `fit()`, `transform()`, `fit_transform()` |
| Pipeline | Menghubungkan langkah-langkah preprocessing dan model menjadi satu objek |
| `coef_` / `intercept_` | Atribut model linear untuk interpretasi |
| `score()` | Mengevaluasi performa model (akurasi/R2) |
| `GridSearchCV` | Optimasi hyperparameter secara exhaustive |
| Metadata Routing | Mengontrol aliran metadata antar komponen pipeline |

### Poin Kunci untuk Diingat:
- Selalu pisahkan data training dan testing SEBELUM menerapkan transformasi.
- Gunakan `fit()` hanya pada data training; gunakan `transform()` pada data testing.
- Pipeline adalah alat yang sangat kuat untuk membuat workflow ML yang bersih, reproducible, dan bebas data leakage.
- scikit-learn dirancang untuk konsistensi: sekali Anda memahami satu estimator, Anda memahami pola dasar semua estimator.
